In [ ]:
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import spacy

from dap_job_quality.utils.keyword_search_patterns import keywords
from dap_job_quality.getters.ojo_getters import get_ojo_sample
from dap_job_quality.utils.spacy_keyword_search import get_matches, get_spans
from dap_job_quality.utils.text_cleaning import clean_text

from dap_job_quality.getters.data_getters import load_s3_jsonl
from dap_job_quality.getters.labelled_data import get_labelled_job_sentences
from dap_job_quality.utils import prodigy_data_utils as pdu

from dap_job_quality import BUCKET_NAME, PROJECT_DIR, config

model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
nlp = spacy.load("en_core_web_sm")

SEED = config["seed"]

def filter_job_ads(labelled_df):
    job_ids = labelled_df["id"].unique()

    # skip the first 10 job ads - we didn't know what we were labelling at that point
    target_ids = job_ids[10:]

    labelled_df_clean = labelled_df[labelled_df["id"].isin(target_ids)]
    # get rid of empty spans
    labelled_df_clean = labelled_df_clean[labelled_df_clean["span"] != ""]
    return labelled_df_clean

In [ ]:
labelled_sents = get_labelled_job_sentences()[0]

labelled_data = pdu.get_spans_and_sentences(labelled_sents)

labelled_df = pd.DataFrame(columns=["span", "sent", "text", "job_id"])

for key in labelled_data.keys():
    temp_df = pd.DataFrame(labelled_data[key])
    temp_df["id"] = int(key)
    labelled_df = pd.concat([labelled_df, temp_df])

labelled_df = labelled_df.drop(["job_id"], axis=1)

labelled_df_clean = filter_job_ads(labelled_df)

labelled_df_clean["sentence"] = labelled_df_clean["sent"].apply(lambda x: x.text)

In [ ]:
def get_negative_example_sentences(labelled_df):
    
    # any sentences already extracted are positive examples
    positive_sentences = labelled_df["sentence"].tolist()
    
    # Find all unique job ads/descs
    unique_job_descs = labelled_df[['id', 'text']].drop_duplicates()
    
    # initialise empty df
    negative_examples_df = pd.DataFrame(columns=['id', 'text', 'sentence'])
    
    for _, row in unique_job_descs.iterrows():
        doc = nlp(row['text'])
        sentences = [sent.text for sent in doc.sents]
        
        for sentence in sentences:
            if sentence not in positive_sentences:
                temp_df = pd.DataFrame({'id': [row['id']], 'text': [row['text']], 'sentence': [sentence]})
                negative_examples_df = pd.concat([negative_examples_df, temp_df])
    
    return negative_examples_df

In [ ]:
negative_examples_df = get_negative_example_sentences(labelled_df_clean)
negative_examples_df["label"] = 0
negative_examples_df.head()

In [ ]:
positive_df = labelled_df_clean[['id', 'sentence', 'span']]
positive_df_short = positive_df.groupby(['id', 'sentence'])['span'].agg(list).reset_index()
positive_df_short.head()

In [ ]:
lookup = pd.read_csv(PROJECT_DIR / "inputs/keyword_lookup - v4.csv")
lookup.head(20)

In [ ]:
categories = list(lookup['subcategory'].unique())

In [ ]:
lookup['target_phrase'].unique()

In [ ]:
def match_sentence(sentence, category, lookup=lookup):
    matches = 0
    target_phrases = lookup[lookup['subcategory']==category]['target_phrase'].tolist()
    for target in target_phrases:
        if target.lower() in sentence.lower():
            # matched_dimension = lookup[lookup['target_phrase'] == target]['subcategory'].values[0]
            matches +=1
            
    if matches > 0:
        return 1
    else:
        return 0

In [ ]:
for category in categories:
    positive_df_short[category] = positive_df_short['sentence'].apply(lambda x: match_sentence(x, category))

In [ ]:
df_with_at_least_one_1 = positive_df_short[positive_df_short[categories].any(axis=1)]

# DataFrame where all of the specified columns are 0
df_with_all_zeros = positive_df_short[positive_df_short[categories].any(axis=1) == False]

In [ ]:
len(df_with_at_least_one_1)

In [ ]:
len(df_with_all_zeros)

In [ ]:
target_embeddings = model.encode(lookup['target_phrase'].tolist(), show_progress_bar=True)

In [ ]:
lookup['embeddings'] = target_embeddings.tolist()

In [ ]:
def get_n_most_similar_phrases(input_sentence, 
                                 lookup,
                                 model,
                                 n: int = 3):
    # most_similar_phrases = {}
    
    input_embedding = model.encode(input_sentence)
    
    similarities = [cosine_similarity([input_embedding], [embed])[0][0] for embed in lookup['embeddings'].apply(pd.Series).values]
    
    top_indices = np.argsort(similarities)[::-1][:n]
    
    similar_phrases = lookup.iloc[top_indices]
    similar_phrases['similarity'] = [similarities[i] for i in top_indices]
       
    return similar_phrases[['dimension', 'subcategory', 'target_phrase', 'similarity']]

In [ ]:
df_with_all_zeros['target_phrase'] = ""
df_with_all_zeros['subcategory'] = ""
df_with_all_zeros['similarity'] = 0

for idx, row in df_with_all_zeros.iterrows():
    # Get the most similar phrases for the current sentence
    temp_df = get_n_most_similar_phrases(row['sentence'], lookup, model, n=1)
    if not temp_df.empty:
        # Assign the results directly to the DataFrame using the index
        df_with_all_zeros.at[idx, 'target_phrase'] = temp_df['target_phrase'].iloc[0]
        df_with_all_zeros.at[idx, 'subcategory'] = temp_df['subcategory'].iloc[0]
        df_with_all_zeros.at[idx, 'similarity'] = temp_df['similarity'].iloc[0]


In [ ]:
df_with_all_zeros.head()

In [ ]:
df_with_all_zeros.to_csv(PROJECT_DIR / "inputs/labelling/data_to_label_manually_for_categories_all_mini_lm_embeddings.csv")